# Stage 1: Train Base Model (VAE/NVAE)

This notebook trains the base representation model. It supports:
1. **Models**: Standard `VAE` or Hierarchical `NVAE`.
2. **Datasets**: `CelebA`, `CIFAR10`, `FFHQ`.

The best model is saved to `checkpoints/best_{model}_{dataset}.pth`.  
This checkpoint will be used in Stage 2 to train the Latent Diffusion Model.


In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms, datasets
import os

# Import custom modules
from training.train_utils import set_seed, Model, VAE, CelebA
from training.nvae_model import NVAE
from training.datasets import FlatImageFolder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data Loader Setup

In [ ]:
def get_dataloader(dataset_name, batch_size=64, image_size=32):
    if dataset_name == 'CelebA':
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        # Assuming ./data/CelebA structure
        dset = CelebA("./data/CelebA", split='train', download=True, transform=transform)
        
    elif dataset_name == 'CIFAR10':
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        dset = datasets.CIFAR10(root='./data/CIFAR10', train=True, download=True, transform=transform)
        
    elif dataset_name == 'FFHQ':
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        dset = FlatImageFolder(root='./data/FFHQ', transform=transform)
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")
        
    return DataLoader(dset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)


## 2. Loss Functions

In [ ]:
def loss_vae(recon, x, mu, logvar, kl_weight=0.00025):
    recon_loss = F.mse_loss(recon, x, reduction='sum') / x.shape[0]
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.shape[0]
    return recon_loss + kl_weight * kl_loss, recon_loss, kl_loss

def loss_nvae(out_dict, x, kl_weight=1.0):
    # Note: Traditional NVAE handles KL per pixel/dim carefully. 
    # Here we use a simplified version for demonstration.
    
    x_rec = out_dict['x_rec']
    recon_loss = F.mse_loss(x_rec, x, reduction='sum') / x.shape[0]
    
    # Global KL: q(z_g|x) vs N(0,1)
    mu_g, log_g = out_dict['mu_g'], out_dict['log_g']
    kl_global = -0.5 * torch.sum(1 + log_g - mu_g.pow(2) - log_g.exp()) / x.shape[0]
    
    # Fine KL: q(z_f|x) vs p(z_f|z_g)
    mu_f, log_f = out_dict['mu_f'], out_dict['log_f']
    mu_p, log_p = out_dict['mu_prior_f'], out_dict['log_prior_f']
    
    # KL(N(mu_f, var_f) || N(mu_p, var_p))
    # = 0.5 * [ log(var_p/var_f) + (var_f + (mu_f - mu_p)^2)/var_p - 1 ]
    # log_term = log_p - log_f
    # var_f = exp(log_f), var_p = exp(log_p)
    
    var_f = torch.exp(log_f)
    var_p = torch.exp(log_p)
    
    kl_fine_element = log_p - log_f + (var_f + (mu_f - mu_p).pow(2)) / var_p - 1.0
    kl_fine = 0.5 * torch.sum(kl_fine_element) / x.shape[0]
    
    total_loss = recon_loss + kl_weight * (kl_global + kl_fine)
    return total_loss, recon_loss, kl_global, kl_fine

## 3. Training Loop

In [ ]:
def train_base_model(model_type, dataset_name, z_dim=64, n_epochs=50):
    save_path = f"checkpoints/best_{model_type}_{dataset_name}.pth"
    os.makedirs("checkpoints", exist_ok=True)
    
    # 1. Setup Model
    # Image size: CIFAR10=32, CelebA=32(cropped), FFHQ=32(resized)
    # You can change input_res=64 if you change transform size
    IMG_RES = 32
    
    if model_type == 'vae':
        model = Model(3, 32, 3, 128, z_dim, 32, 32).to(device)
    elif model_type == 'nvae':
        model = NVAE(z_dim=z_dim, input_res=IMG_RES).to(device)
    else:
        raise ValueError("Unknown model")
        
    # 2. Setup Data
    dataloader = get_dataloader(dataset_name, batch_size=128, image_size=IMG_RES)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_loss = float('inf')
    
    print(f"Starting training for {model_type} on {dataset_name}...")
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        steps = 0
        
        for images, _ in dataloader:
            images = images.to(device)
            
            optimizer.zero_grad()
            
            if model_type == 'vae':
                mu, logvar = model.encoder(images)
                z = model.reparameterize(mu, logvar)
                recon = model.decoder(z)
                loss, r_loss, k_loss = loss_vae(recon, images, mu, logvar)
            else: # nvae
                out = model(images)
                loss, r_loss, kg, kf = loss_nvae(out, images)
                
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            steps += 1
            
        avg_loss = epoch_loss / steps
        print(f"Epoch {epoch+1}: Loss={avg_loss:.4f} (Saved: {avg_loss < best_loss})")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), save_path)
            
    print(f"Done! Best model saved to {save_path}")
    return model

In [ ]:
if __name__ == "__main__":
    set_seed(42)
    # Example: Train VAE on CelebA
    # train_base_model('vae', 'CelebA', n_epochs=50)
    
    # Example: Train NVAE on CIFAR10 (for quick test)
    train_base_model('nvae', 'CIFAR10', n_epochs=10)
    
    # Uncomment to train others:
    # train_base_model('nvae', 'CelebA', n_epochs=50)
    # train_base_model('vae', 'FFHQ', n_epochs=50)